<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/Spark_SQL_Find_Your_Perfect_Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🌍 Finding Your Perfect Weather with Spark

In this notebook, you will build a simple data system to answer a real question:

> **Which city has the best weather based on your preferences?**

Different people want different things:

- ☀️ warm and sunny  
- 🪁 windy  
- 🌧️ little or no rain  

Your goal is to **use data to recommend the best city**.

## Your mission

By the end of this notebook, you will be able to:

- find warm and sunny days  
- compare cities based on weather  
- define your own “perfect weather”  

👉 and use Spark to find the best match.

## Getting Started

We begin by collecting real weather data from an API.

An API (Application Programming Interface) allows us to request data from an external service and receive it in **JSON format**.

👉 Below, we define a function `get_weather` that:
- connects to the API  
- requests weather data  
- returns the result for a given city (using latitude and longitude)

In [ ]:
import requests

def get_weather(city, latitude, longitude):
    # API endpoint (where we send the request)
    url = "https://api.open-meteo.com/v1/forecast"

    # Parameters sent to the API
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_probability_max",
            "windspeed_10m_max"
        ],
        "timezone": "auto",
        "forecast_days": 16,
        "past_days": 92
    }

    # Send the request to the API
    response = requests.get(url, params=params)

    # Convert the response to JSON format (Python dictionary)
    data = response.json()

    # Add the city name to the data (for later use in Spark)
    data["city"] = city

    # Return the result
    return data

### Get Data for One City

Let’s call the function for one city and store the result.

In [ ]:
data = get_weather("Madrid", 40.41, -3.7)

### Inspect the Result

Let’s look at the data returned by the API.

👉 In a notebook, you can display the content of a variable by simply writing its name and running the cell.

In [ ]:
data

# ⚡ Starting Spark

Before working with the data, we need to start a Spark session.

This is the entry point for working with Spark.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Spark SQL Intro") \
    .getOrCreate()

## Loading the Data into Spark

We received the data in JSON format.

Now, we will load it into Spark so we can explore its structure and work with it more easily.

In [ ]:
import json
df = spark.read.json(
    spark.sparkContext.parallelize([json.dumps(data)])
)

> **Note**
>
> The data we received from the API is a Python object (a dictionary).
>
>Spark cannot work directly with Python variables, so we need to convert it into a format Spark understands.
>
>👉 In simple terms:
>- `json.dumps(...)` converts the data to text  
>- `parallelize(...)` sends it to Spark  
>- `read.json(...)` turns it into a table  
>
>You don’t need to fully understand every part of this yet.

## Inspecting the Data

Now that the data is loaded into Spark, let’s explore its structure.

Spark represents data using a **schema**, which describes how the data is organized.

In [ ]:
# ✍️ Exercise:
# Show the schema of the DataFrame we created above
# write your code below:


In [ ]:
# ✍️ Exercise:
# Display the content of the DataFrame
# write your code below:


In [ ]:
# ✍️ Exercise:
# Display the DataFrame in full (no truncation e.g., truncate=False) and in vertical format
# write your code below:



## Flattening the Data Step by Step

The weather data is still nested, so it is not yet in a table format.

To understand how Spark transforms this data, we start with a simple step:

👉 flatten a single column.

We will use the `time` values inside the `daily` field to create one row per day.

In [ ]:
# ✍️ Exercise:
# Use explode to flatten the column "daily.time"
# Create a new DataFrame with:
# - the city
# - one date per row
# Display the result
from pyspark.sql.functions import explode
# write your code below




>👉 Each row now represents a single date.
>
>This is the first step in transforming nested data into a table.

## Understanding arrays_zip function

Before applying this to our weather data, let’s look at a simple example.

We will create a small DataFrame with two lists:
- dates
- temperatures

Then we will see how to combine them correctly.

In [ ]:
from pyspark.sql import Row

example_data = [
    Row(
        city="ExampleCity",
        dates=["Day1", "Day2", "Day3"],
        temps=[20, 25, 22]
    )
]

df_example = spark.createDataFrame(example_data)
df_example.show(truncate=False)

df_example.printSchema()

+-----------+------------------+------------+
|city       |dates             |temps       |
+-----------+------------------+------------+
|ExampleCity|[Day1, Day2, Day3]|[20, 25, 22]|
+-----------+------------------+------------+

root
 |-- city: string (nullable = true)
 |-- dates: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- temps: array (nullable = true)
 |    |-- element: long (containsNull = true)



Let’s see what happens if we explode both columns directly.

In [ ]:
from pyspark.sql.functions import explode

df_wrong = df_example.select(
    "city",
    explode("dates").alias("date"),
    explode("temps").alias("temp")
)

df_wrong.show()

+-----------+----+----+
|       city|date|temp|
+-----------+----+----+
|ExampleCity|Day1|  20|
|ExampleCity|Day1|  25|
|ExampleCity|Day1|  22|
|ExampleCity|Day2|  20|
|ExampleCity|Day2|  25|
|ExampleCity|Day2|  22|
|ExampleCity|Day3|  20|
|ExampleCity|Day3|  25|
|ExampleCity|Day3|  22|
+-----------+----+----+



> ⚠️ **Problem**
>
>The dates and temperatures are not matched correctly.
>
>👉 Spark creates all possible combinations instead of keeping values aligned  
>(this is similar to a cross join).

Now let’s fix this using `arrays_zip`.

In [ ]:
from pyspark.sql.functions import arrays_zip

df_correct = df_example.select(
    "city",
    explode(arrays_zip("dates", "temps")).alias("data")
).select(
    "city",
    "data.dates",
    "data.temps"
)

df_correct.show()

+-----------+-----+-----+
|       city|dates|temps|
+-----------+-----+-----+
|ExampleCity| Day1|   20|
|ExampleCity| Day2|   25|
|ExampleCity| Day3|   22|
+-----------+-----+-----+



**Now it works correctly**

Each date is matched with the correct temperature.

### How `arrays_zip` + `explode` work together

#### 1. `arrays_zip(...)`

This function combines multiple lists **position by position**.

Example:

    dates = ["Day1", "Day2", "Day3"]
    temps = [20, 25, 22]

👉 After `arrays_zip`:

    [
      {dates: "Day1", temps: 20},
      {dates: "Day2", temps: 25},
      {dates: "Day3", temps: 22}
    ]

👉 Values are now correctly paired.

---

### 2. `explode(...)`

The `explode` function creates **one row per element** of a list.

So:

    explode(arrays_zip(...))

👉 becomes:

| date | temp |
|------|------|
| Day1 | 20   |
| Day2 | 25   |
| Day3 | 22   |

---

### Summary

- `arrays_zip` → keeps values aligned  
- `explode` → creates rows  

👉 Together, they transform nested lists into a clean table.

### Applying This to Our Data

Now that we understand how `arrays_zip` works, let’s apply it to our weather data.

Previously, we flattened only the dates.  
Now, we also want to keep the **maximum temperature for each day**.

👉 Each row should contain:
- the city  
- the date  
- the maximum temperature  

---

### ✍️ Exercise

Use `arrays_zip` and `explode` to:
- keep the values aligned  
- create one row per day

In [ ]:
# ✍️ Exercise:
# Use arrays_zip and explode to create a new DataFrame with:
# - city
# - date
# - maximum temperature
# Display the result

# 💡 Hint:
# The columns you need are:
# - "daily.time"
# - "daily.temperature_2m_max"

from pyspark.sql.functions import arrays_zip, explode

# write your code below


## 🌦️ Building the Full Weather Table

So far, we created a table with the **date** and the **maximum temperature**.

Now we will use the same idea to include more weather attributes for each day:

- date
- maximum temperature
- minimum temperature
- precipitation probability
- wind speed

👉 Each row should represent the weather for one city on one day.

In [ ]:
# ✍️ Exercise:
# Create a DataFrame with one row per day including:
# - city
# - date
# - temperature_2m_max
# - temperature_2m_min
# - precipitation_probability_max
# - windspeed_10m_max
# Display the result

# 💡 Hint:
# Use arrays_zip with all the daily columns

from pyspark.sql.functions import arrays_zip, explode

# write your code below


### Inspect the Result

Let’s inspect the DataFrame we just created.

In [ ]:
# ✍️ Exercise:
# Show the schema and display the DataFrame

# write your code below


root
 |-- city: string (nullable = true)
 |-- time: string (nullable = true)
 |-- temperature_2m_max: double (nullable = true)
 |-- temperature_2m_min: double (nullable = true)
 |-- precipitation_probability_max: long (nullable = true)
 |-- windspeed_10m_max: double (nullable = true)

+------+----------+------------------+------------------+-----------------------------+-----------------+
|  city|      time|temperature_2m_max|temperature_2m_min|precipitation_probability_max|windspeed_10m_max|
+------+----------+------------------+------------------+-----------------------------+-----------------+
|Madrid|2026-01-10|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-11|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-12|              NULL|              NULL|                            0|             NULL|
|Madrid|2026-01-13|              NULL|              NULL|                     

# Working with Multiple Cities

So far, we have prepared weather data for a single city.

Now, we will repeat the same process for several cities and combine the results into one DataFrame.

👉 This will allow us to compare cities using the same weather criteria.

We will use the following list of cities:

In [ ]:
cities = [
    ("Madrid", 40.41, -3.70),
    ("Paris", 48.85, 2.35),
    ("Berlin", 52.52, 13.41),
    ("Rome", 41.90, 12.49),
    ("London", 51.50, -0.12),
    ("Barcelona", 41.38, 2.17),
    ("Lisbon", 38.72, -9.14),
    ("Amsterdam", 52.37, 4.90),
    ("Brussels", 50.85, 4.35),
    ("Vienna", 48.21, 16.37),
    ("Prague", 50.08, 14.43),
    ("Copenhagen", 55.68, 12.57),
    ("Dublin", 53.35, -6.26),
    ("Athens", 37.98, 23.72),
    ("Zurich", 47.37, 8.54)
]

### Exercise

Create a single DataFrame containing weather data for all cities.

Steps:
1. Loop over the list of cities
2. Call `get_weather`
3. Load the result into Spark
4. Flatten the nested data
5. Combine all results into one DataFrame

👉 Reuse the code from the previous sections.

**Complete the `TODO` parts in the code below.**

In [ ]:
from pyspark.sql.functions import arrays_zip, explode
import json

# We will store the DataFrame for each city in this list
dfs = []

# Loop over all cities in the list
# Each element contains: (city name, latitude, longitude)
for city, lat, lon in cities:

    # -----------------------------
    # Step 1: Get data from the API
    # -----------------------------
    # For each city, we call the function to retrieve weather data
    data = get_weather(city, lat, lon)

    # -----------------------------
    # Step 2: Load data into Spark
    # -----------------------------
    # Convert the JSON data into a Spark DataFrame
    df_temp = spark.read.json(
        spark.sparkContext.parallelize([json.dumps(data)])
    )

    # -----------------------------
    # Step 3: Flatten the data
    # -----------------------------
    # We transform the nested structure into a flat table
    # Each row will represent one day for the current city
    df_temp = df_temp.select(
        "city",
        explode(
            arrays_zip(
                # TODO: add the same columns as before
            )
        ).alias("weather")
    ).select(
        "city",
        # TODO: extract the fields from "weather"
    )

    # -----------------------------
    # Step 4: Store the result
    # -----------------------------
    # We add the DataFrame for this city to the list
    dfs.append(df_temp)

# -----------------------------
# Step 5: Combine all DataFrames
# -----------------------------
# At this point, we have one DataFrame per city in the list "dfs"

# Start with the first DataFrame
weather_df = dfs[0]

# Loop through the remaining DataFrames
# and combine them using union (stacking rows)
for d in dfs[1:]:
    weather_df = weather_df.union(d)

### Inspect the Result

Let’s inspect the DataFrame we just created.

In [ ]:
# ✍️ Exercise:
# Show the first rows of the combined DataFrame

# write your code below


👉 We now have a dataset with multiple cities.

Each row represents:
- one city
- one day
- its weather conditions

Next, we will use SQL to analyze and compare cities.

### Renaming Columns

The column names are long and not very easy to read.

Let’s rename them to make our queries simpler.

In [ ]:
# ✍️ Exercise:
# Rename the columns to:
# - time -> date
# - temperature_2m_max -> temp_max
# - temperature_2m_min -> temp_min
# - precipitation_probability_max -> rain_prob
# - windspeed_10m_max -> wind_speed

# write your code below



👉 The table is now easier to read.

Each row represents:
- one city
- one day
- multiple weather measurements

We are now ready to analyze the data using SQL.

# Analyzing the Data with SQL

Now that we have a clean dataset with multiple cities, we can start analyzing it.

We will use **SQL in Spark** to answer questions about the weather.

👉 First, we register the DataFrame as a table.

In [ ]:
weather_df.createOrReplaceTempView("weather")

### 🔎 Exploring the Data

Before writing more advanced queries, let’s explore the dataset.

👉 These quick queries will help us understand the size and structure of the data.

### 📊 Exercise 1 — How many rows?

Find the total number of rows in the dataset.

In [ ]:
# ✍️ Exercise:
# Count the total number of rows in the table

# write your code below
spark.sql("""

""").show()

### 🏙️ Exercise 2 — How many cities?

Find how many different cities are in the dataset.

In [ ]:
# ✍️ Exercise:
# Count the number of distinct cities

# write your code below
spark.sql("""

""").show()

### 📅 Exercise 3 — How many months?

Find how many different months are present in the dataset.

In [ ]:
# ✍️ Exercise:
# Count the number of distinct months in the data

# write your code below
spark.sql("""

""").show()

### 📆 Exercise 4 — Which months are included?

List the months present in the dataset.

In [ ]:
# ✍️ Exercise:
# list the months included

# write your code below
spark.sql("""

""").show()

### ☀️ Challenge 1 — Find Warm and Dry Days

Find all days where:
- the maximum temperature is at least 20°C  
- the probability of rain is less than 20%  

👉 Display the result.

In [ ]:
# ✍️ Exercise:
# Write a SQL query to find warm and dry days

# write your code below
spark.sql("""

""").show()

### 🏆 Challenge 2 — Which City Has the Most Nice Days?

Using the same definition as before (warm and dry):

👉 Count how many "nice days" each city has  
👉 Sort the results from highest to lowest

In [ ]:
# ✍️ Exercise:
# Write a SQL query to rank cities by number of nice days

# write your code below


### 🪁 Challenge 3 — Windy Days

Find the cities with the most windy days.

A day is considered **windy** if:
- wind speed is between 15 and 30  
- probability of rain is less than 20%  

👉 Count how many windy days each city has and rank them.

In [ ]:
# ✍️ Exercise:
# Write a SQL query to find and rank cities by number of windy days

# write your code below


### 🎯 Challenge 4 — Your perfect weather

Now it's your turn!

👉 Define your own "perfect weather" using:
- temperature
- rain probability
- wind speed

Then:
- find matching days
- rank the cities

Be creative!

In [ ]:
# ✍️ Exercise:
# Define your own perfect weather and rank the cities

# write your code below


### 📅 Challenge 5 — Best Month to Visit

So far, we have analyzed weather day by day.

But in real life, people plan trips by **month**, not by individual days.

👉 Let’s find out:

**Which is the best month to visit each city?**

### Your task

1. Convert the `date` column to a proper date format  
2. Extract the year and month  
3. Calculate the **average maximum temperature per city per month**  
4. Sort the results to identify the warmest months  


👉 Use the following functions:
- `to_date()`
- `month()`
- `year()`

In [ ]:
# ✍️ Exercise:
# Analyze the average maximum temperature per city per month
from pyspark.sql.functions import to_date, month, year, avg
# write your code below


### 📊 Bonus Challenge — Which City Has the Most Consistent Weather?

A city can be pleasant not only because it is warm, but also because its weather is stable.

👉 We will use a new function: `STDDEV_POP`

### What does it do?

- It measures how much values vary from the average  
- A **low value** means the temperature is stable (consistent)  
- A **high value** means the temperature changes a lot  

👉 In our case:
- low variation = more predictable weather  

### Your task

Find the cities with the most consistent maximum temperature.


In [ ]:
# ✍️ Bonus Exercise:
# Find the cities with the most consistent maximum temperature

# write your code below

